In [6]:
import sys
sys.path.append('/home/potzschf/repos/')
from helperToolz.helpsters import *
from helperToolz.dicts_and_lists import *
from helperToolz.guzinski import * 
import geopandas as gpd
from collections import defaultdict
import zipfile
from concurrent.futures import ProcessPoolExecutor

In [7]:
# zip masked chips
unmasked_folder = '/data/Aldhani/eoagritwin/fields/04_Predictions/GERMANY/FromScratch_IACS_dilate_True_BorderEdgeCutted_RGB_NDVI_exclude_True_with_overlap_47/2023/chips_folder/unmasked_chips/'
mfiles = getFilelist(unmasked_folder, '.tif')

In [ ]:


print(len(mfiles))
n = 8
chunk_size = math.ceil(len(mfiles) / n)
chunks = [mfiles[i:i + chunk_size] for i in range(0, len(mfiles), chunk_size)]


def zip_chunk(args):
    idx, chunk, masked_folder = args

    zip_name = os.path.join(masked_folder, f"unmasked_{idx}.zip")

    with zipfile.ZipFile(
        zip_name,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as zf:
        for tif_path in chunk:
            zf.write(tif_path, arcname=os.path.basename(tif_path))

print('zipping masked chips')
with ProcessPoolExecutor(max_workers=30) as executor:
    executor.map(
        zip_chunk,
        [(idx, chunk, unmasked_folder) for idx, chunk in enumerate(chunks)]
    )



106299
zipping masked chips


In [ ]:

##########################################################################

mask = (arr_field != 0) & (~np.isnan(arr_field)) & (~np.isnan(arr_b1))
flat_field = arr_field[mask].astype(int)
flat_b1 = arr_b1[mask]

# Create DataFrame
df = pd.DataFrame({'FieldID': flat_field, 'B1': flat_b1})

# Compute median per field
median_df = df.groupby('FieldID', as_index=False)['B1'].median()
median_df.rename(columns={'B1': 'B1_median'}, inplace=True)
median_df.to_csv(path_safe(f"{origin}output/uncertainty/{aoi}_mean_b1_score_per_field_ext{ext}_bound{bound}.csv"), index=False)

# Create a lookup map
median_map = dict(zip(median_df['FieldID'], median_df['B1_median']))

# Fill raster
arr_field_fill = np.vectorize(median_map.get)(arr_field)
# get_value = lambda x: median_map.get(x, np.nan)
# arr_field_fill = np.vectorize(get_value, otypes=[float])(arr_field)

In [ ]:
arr_expo = np.where(arr_field_fill == None, np.nan, arr_field_fill)


In [ ]:
arr_export = arr_export.astype(np.float32)
arr_export[236:-2766,1416:] = arr_expo

npTOdisk(arr_export, path_fields,
         f"{origin}output/uncertainty/{aoi}_Fields_MEDIAN_B1_SCORE_ext{ext}_bound{bound}.tif",
         noData=0, d_type=gdal.GDT_Float32)


In [ ]:
print(ext_b1)
print(ext_fields)

In [ ]:
print((ext_b1[3] - ext_fields[3])/10)
print((ext_b1[1] - ext_fields[1])/10)

print((ext_fields[2] - ext_b1[2])/10)
print((ext_b1[0] - ext_fields[0])/10)

In [ ]:
tt = arr_field[236:-2766,1416:]
print(tt.shape)
qq = arr_b1[:,:-69]
print(qq.shape)

In [21]:
# create a mask for clean fields for uncertainty maps
pathi = f"{origin}output/uncertainty/{aoi}_Fields_MEDIAN_B1_SCORE_ext{ext}_bound{bound}.tif"
pathi2 = f"{origin}output/uncertainty/{aoi}_Borders_MEDIAN_B2_SCORE_ext{ext}_bound{bound}.tif"
ds = gdal.Open(pathi)
ds2 = gdal.Open(pathi2)
arr = ds.GetRasterBand(1).ReadAsArray()
arr2 = ds2.GetRasterBand(1).ReadAsArray()
arr_export = zeros = np.zeros_like(arr)
arr = arr[236:-2766,1416:]
arr2 = arr2[:,:-69]
# arr_np = np.where(np.logical_or(arr == 0, np.isnan(arr)), 1, 0)
mask1 = (arr == 0) | np.isnan(arr)
mask2 = (arr2 == 0) | np.isnan(arr2)
arr_np = np.where(mask1 & mask2, 1, 0)

arr_export = arr_export.astype(np.float32)
arr_export[236:-2766,1416:] = arr_np
npTOdisk(arr_export, pathi,
         f"{origin}output/uncertainty/MASK_ext{ext}_bound{bound}.tif",
         noData=0, d_type=gdal.GDT_Byte)

In [14]:
ext_bounds = getExtentRas(pathi2)
ext_fields = getExtentRas(pathi)

print(ext_bounds)
print(ext_fields)

{'Xmin': 4030286.3630416505, 'Xmax': 4674566.3630416505, 'Ymin': 2683979.6079648044, 'Ymax': 3552459.6079648044}
{'Xmin': 4016126.3630416505, 'Xmax': 4673876.3630416505, 'Ymin': 2656319.6079648044, 'Ymax': 3554819.6079648044}


In [13]:
print((ext_bounds['Xmin'] - ext_fields['Xmin']) / 10)
print((ext_bounds['Xmax'] - ext_fields['Xmax']) / 10)
print((ext_bounds['Ymin'] - ext_fields['Ymin']) / 10)
print((ext_bounds['Ymax'] - ext_fields['Ymax']) / 10)


1416.0
69.0
2766.0
-236.0


In [15]:
arr[236:-2766,1416:].shape

(86848, 64359)

In [18]:
arr2[:,:-69].shape

(86848, 64359)

In [ ]:
# cut em all into smaller pieces for easier map creation
field_path = f"{origin}output/uncertainty/{aoi}_Fields_MEDIAN_B1_SCORE_ext{ext}_bound{bound}.tif"
border_path = f"{origin}output/uncertainty/{aoi}_Borders_MEDIAN_B2_SCORE_ext{ext}_bound{bound}.tif"
mask_path = f"{origin}output/uncertainty/MASK_ext{ext}_bound{bound}.tif"

evaps = 

In [ ]:
subset_mask_to_prediction_extent

In [ ]:
from FieldWaterUseTools.FuncBox.Misc import getFilelist, path_safe


year = 2023
master = f"{origin}fields/04_Predictions/GERMANY/FromScratch_IACS_dilate_True_BorderEdgeCutted_RGB_NDVI_exclude_True_with_overlap_47/{year}/"
chips_folder = f"{master}chips_folder/unmasked_chips/"
masked_folder = path_safe(f"{master}chips_folder/masked_chips/")
files = getFilelist(chips_folder, '.tif')
ds = gdal.Open(f"{master}vrt/Masked_THUENEN_CTM_2023.tif")

In [ ]:
conti = []

for file in files:
    # load masked stack
    chip = ds.GetRasterBand(1).ReadAsArray(
        xoff=int(file.split('X_')[-1].split('_')[0]),
        yoff=int(file.split('X_')[-1].split('_')[2].split('.')[0]),
        win_xsize=236,  
        win_ysize=236 
    )

    if np.nansum(chip) > 0:

        with rasterio.open(file) as src:
            bounds = src.bounds  # left, bottom, right, top
            geom = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
            crs = src.crs

            conti.append({
                "filename": os.path.basename(file),
                "geometry": geom,
                "crs_used": str(crs)
            })

            chip2 = ds.GetRasterBand(2).ReadAsArray(
                xoff=int(file.split('X_')[-1].split('_')[0]),
                yoff=int(file.split('X_')[-1].split('_')[2].split('.')[0]),
                win_xsize=236,  
                win_ysize=236 
            )

            chip3 = ds.GetRasterBand(3).ReadAsArray(
                xoff=int(file.split('X_')[-1].split('_')[0]),
                yoff=int(file.split('X_')[-1].split('_')[2].split('.')[0]),
                win_xsize=236,  
                win_ysize=236 
            )

            stack = np.stack([chip, chip2, chip3], axis=0)

            with rasterio.open(
                f"{masked_folder}chips_masked256{file.split('_unmasked256')[-1]}",
                "w",
                driver="GTiff",
                height=stack.shape[1],
                width=stack.shape[2],
                count=stack.shape[0],
                dtype=stack.dtype,
                crs=crs,          # z.B. von einem Referenz-Datensatz: ref_ds.crs
                transform=src.transform  # z.B. ref_ds.transform
            ) as dst:
                dst.write(stack)

In [ ]:
gdf = gpd.GeoDataFrame(conti, geometry="geometry", crs=conti[0]["crs_used"])
gdf.to_file(f"{master}{year}_grid_tiles.gpkg", driver="GPKG", layer="tiles")

In [ ]:
from FieldWaterUseTools.FuncBox.FieldFuncis import predicted_chips_to_vrt

predicted_chips_to_vrt(f"{master}chips_folder/", 'masked_chips', 256, 20, path_safe(f"{master}vrt/"), pyramids=True)

In [ ]:
# zip em
mfiles = getFilelist(masked_folder, '.tif')
print(len(mfiles))
n = 8
chunk_size = math.ceil(len(mfiles) / n)
chunks = [mfiles[i:i + chunk_size] for i in range(0, len(mfiles), chunk_size)]

In [ ]:
len(chunks[0])

In [ ]:
from concurrent.futures import ProcessPoolExecutor

def zip_chunk(args):
    idx, chunk, masked_folder = args

    zip_name = os.path.join(masked_folder, f"masked_{idx}.zip")

    with zipfile.ZipFile(
        zip_name,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as zf:
        for tif_path in chunk:
            zf.write(tif_path, arcname=os.path.basename(tif_path))



with ProcessPoolExecutor(max_workers=30) as executor:
    executor.map(
        zip_chunk,
        [(idx, chunk, masked_folder) for idx, chunk in enumerate(chunks)]
    )